In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TOP_N_CUISINES = 10
TOP_N_CITIES = 20

# ---------- Load data ----------
DATA_PATH = Path("..") / "Dataset .csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH.resolve()}")
if DATA_PATH.stat().st_size == 0:
    raise ValueError("Dataset file is empty. Copy the real file again.")

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
print("Shape:", df.shape)

# ---------- Validate columns ----------
YES_NO_COLS = ["Has Table booking", "Has Online delivery", "Is delivering now"]
NUMERIC_COLS = [
    "Votes",
    "Average Cost for two",
    "Price range",
    "Longitude",
    "Latitude",
    "Country Code",
    "Aggregate rating",
]
REQUIRED = ["Cuisines", "City"] + YES_NO_COLS + NUMERIC_COLS
missing_cols = [c for c in REQUIRED if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

print(df[REQUIRED].isnull().sum())

# ---------- Clean ----------
df = df.drop_duplicates().copy()

# Target = the first (primary) cuisine listed. Rows with no cuisine are dropped.
df = df.dropna(subset=["Cuisines"]).copy()
df["Primary Cuisine"] = df["Cuisines"].str.split(",").str[0].str.strip()

# Keep only the most common cuisines so every class has enough examples.
top_cuisines = df["Primary Cuisine"].value_counts().head(TOP_N_CUISINES).index
df = df[df["Primary Cuisine"].isin(top_cuisines)].copy()
print("\nCuisine class counts:")
print(df["Primary Cuisine"].value_counts())

# ---------- Encode ----------
for col in YES_NO_COLS:
    df[col] = df[col].astype(str).str.strip().str.title().map({"Yes": 1, "No": 0})
    if df[col].isnull().any():
        raise ValueError(f"Unexpected values in column '{col}'")

for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# City has many values, so keep the top cities and group the rest as "Other".
top_cities = df["City"].value_counts().head(TOP_N_CITIES).index
df["City Group"] = np.where(df["City"].isin(top_cities), df["City"], "Other")

FEATURES = NUMERIC_COLS + YES_NO_COLS
df = df.dropna(subset=FEATURES).copy()

X = pd.get_dummies(df[FEATURES + ["City Group"]], columns=["City Group"], dtype=int)
y = df["Primary Cuisine"]
print("\nFinal data:", X.shape)

# ---------- Split (stratified keeps class proportions) ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

# ---------- Train and evaluate ----------
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
        ),
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    results.append(
        {
            "Model": name,
            "Accuracy": round(accuracy_score(y_test, pred), 4),
            "Precision (macro)": round(
                precision_score(y_test, pred, average="macro", zero_division=0), 4
            ),
            "Recall (macro)": round(
                recall_score(y_test, pred, average="macro", zero_division=0), 4
            ),
            "Precision (weighted)": round(
                precision_score(y_test, pred, average="weighted", zero_division=0), 4
            ),
            "Recall (weighted)": round(
                recall_score(y_test, pred, average="weighted", zero_division=0), 4
            ),
        }
    )

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
print("\n", results_df.to_string(index=False))

best_name = results_df.iloc[0]["Model"]
best_pred = predictions[best_name]
print("\nBest model:", best_name)

# ---------- Per-cuisine performance ----------
print("\n", classification_report(y_test, best_pred, zero_division=0))

report = classification_report(y_test, best_pred, zero_division=0, output_dict=True)
per_class = pd.DataFrame(report).T.loc[sorted(y.unique()), ["precision", "recall", "support"]]

# ---------- Plots ----------
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, xticks_rotation=45, cmap="Blues", ax=ax, colorbar=False
)
ax.set_title(f"Confusion matrix ({best_name})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

per_class[["precision", "recall"]].plot(kind="bar", figsize=(9, 5))
plt.title("Precision and recall per cuisine")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_cuisine_performance.png", dpi=150)
plt.show()

results_df.to_csv(OUTPUT_DIR / "model_results.csv", index=False)
per_class.to_csv(OUTPUT_DIR / "per_cuisine_results.csv")
print("Saved results to", OUTPUT_DIR.resolve())